# 00 — Talking to Models

**Prerequisites:** 
- Python 3.10+ with `requests` installed
- Ollama running locally with at least one model pulled
- Recommended: 8+ GB VRAM for responsive performance
  
If you don't have Ollama set up yet, complete **notebook 01** first, then return here.
***********************************************************************************************

You've used REST APIs. LLM APIs look the same on the wire — POST a JSON body, get a JSON response. What's different is everything around them: the response is non-deterministic, the input format is conversational, and your prompt is now part of the program's behavior.

This notebook gets you fluent enough with that to read the rest of the course code and know what's going on. It's not a survey of LLMs. It's the working knowledge you need before notebook 02, where we stop calling APIs by hand and start using `llm_engines`.

## The wire format

Every modern LLM API — Ollama, Anthropic, OpenAI, vLLM — accepts the same basic shape:

```json
{
  "model": "qwen3.5:9b",
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user",   "content": "..."}
  ],
  "temperature": 0.7
}
```

Three roles, in order: `system` sets behavior for the whole conversation, `user` and `assistant` alternate as the dialogue. Send the entire history every call — the model is stateless. The fact that conversations *feel* continuous is your application maintaining the message list, not the model remembering anything.

Run this. Make sure Ollama is up (`ollama serve`) and you've pulled the model:

In [1]:
import requests

OLLAMA = "http://localhost:11434/v1/chat/completions"
MODEL  = "qwen3.5:9b"   # adjust to whatever you pulled

def chat(messages, *, temperature=0.7, model=MODEL, max_tokens=512) -> str:
    r = requests.post(
        OLLAMA,
        json={
            "model": model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "reasoning_effort": "none",
        },
        timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    return data["choices"][0]["message"].get("content", "")

print(chat([
    {"role": "system", "content": "You are terse. One sentence answers."},
    {"role": "user",   "content": "What does HTTP stand for?"},
]))

That's the whole API. Everything else in this notebook is about getting useful output out of that one call.

**Thinking modes:** Models like Qwen3.6 can generate internal reasoning before responding. 
That's useful for hard problems but wastes time on simple examples. Set `reasoning_effort: "none"` 
to disable it. Production code should make this configurable per request.

## Tokens and what they cost you

Models don't see characters or words — they see *tokens*, chunks of roughly 3-4 characters in English. `"unbelievable"` might be one token; `"unbelievably"` might be three. You don't need to predict the split, but you need to know two things:

**1. Tokens are your budget.** Every model has a context window — the maximum number of tokens (prompt + response) it can handle in one call. Currently common ceilings:

- `qwen3.5:9b` — 128K tokens (~96K words)
- `qwen3.6:27b` — 128K
- `llama4:scout` — 10M (genuinely useful for whole-codebase reasoning)

**2. Cloud APIs charge per token.** Local models cost electricity. This shapes architecture: if you're sending 50K tokens of context per request and you make 10K requests a day, the difference between `qwen3.5:9b` running on your own GPU and Claude Opus is roughly $0 vs several hundred dollars daily.

Quick estimate: divide character count by 4. Close enough for budgeting.

## Temperature

Temperature is the only generation parameter you'll touch constantly. It controls how the model samples from its output probability distribution.

| Value | What you get | Use it for |
|---|---|---|
| `0.0` | Deterministic — same input, same output | Extraction, classification, code generation |
| `0.3-0.7` | Slight variation, mostly stable | Q&A, summarization, the default |
| `0.8-1.2` | Genuinely creative output | Brainstorming, writing |
| `> 1.5` | Chaotic | Almost never |

Set it explicitly. Don't trust defaults — they vary by provider, and silent variation in supposedly-deterministic code wastes a lot of debugging time.

In [2]:
import time

question = "Give me a name for a Python library that profiles memory leaks."

for i, t in enumerate((0.0, 0.7, 1.2), 1):
    print(f"Starting call {i}/3 at temperature {t}...")
    start = time.time()
    answer = chat(
        [{"role": "user", "content": question}],
        temperature=t,
    )
    elapsed = time.time() - start
    print(f"[t={t}, {elapsed:.1f}s] {answer.strip()[:120]}\n")

## System prompts are the lever

Most prompt engineering is system prompt engineering. The system message is read by the model as standing instructions — it shapes tone, format, and the kind of failures the model will make. Specific beats clever.

Compare:

In [3]:
vague = chat([
    {"role": "system", "content": "You are helpful."},
    {"role": "user",   "content": "Is this code thread-safe? def increment(): counter[0] += 1"},
])

specific = chat([
    {"role": "system", "content":
     "You review Python code for concurrency issues. Identify the specific bug, name the "
     "primitive that would fix it, and show the corrected code. No throat-clearing."},
    {"role": "user", "content": "Is this code thread-safe? def increment(): counter[0] += 1"},
])

print("VAGUE:\n", vague[:400])
print("\nSPECIFIC:\n", specific[:400])

The vague version produces a meandering essay. The specific version produces something you can drop into a code review. The model didn't get smarter — you stopped making it guess what you wanted.

## Few-shot: show, don't describe

When you need a specific output format, examples beat instructions. Build the format into the conversation history as if previous turns had already happened, then ask for the next one.

This works because the model is fundamentally a next-token predictor — it pattern-matches on what came before. Give it three examples of the pattern and the fourth one comes out the same shape.

In [4]:
# Classify support tickets. No description of the format — the examples are the spec.
result = chat([
    {"role": "system", "content": "Classify support tickets."},
    {"role": "user",      "content": "My order hasn't arrived in 3 weeks."},
    {"role": "assistant", "content": "shipping | high"},
    {"role": "user",      "content": "Charged twice for the same item."},
    {"role": "assistant", "content": "billing | high"},
    {"role": "user",      "content": "How do I change my email address?"},
    {"role": "assistant", "content": "account | low"},
    {"role": "user",      "content": "App crashes when I open the settings page."},
], temperature=0.0)

print(result)

Three examples is usually enough. More than five rarely helps and starts eating context budget.

## Chain-of-thought: free accuracy on reasoning tasks

Models are dramatically better at multi-step problems when you ask them to work through the steps. This isn't a trick — they actually use the intermediate tokens as scratch space. The output before the final answer is doing real computation.

The pattern: append `"Think step by step before giving your final answer."` to the user message. That's it. The accuracy lift on math, logic, and code review is consistent enough that it's worth doing by default for anything non-trivial.

In [5]:
# Without CoT — model often skips a step
fast = chat([
    {"role": "user", "content":
     "A regex `^[a-z]+$` is matched against 'Hello'. Does it match? Yes or no."}
], temperature=0.0)

# With CoT — model walks through the constraints
slow = chat([
    {"role": "user", "content":
     "A regex `^[a-z]+$` is matched against 'Hello'. Does it match?\n"
     "Think step by step before giving your final answer."}
], temperature=0.0)

print("FAST:", fast)
print("\nSLOW:", slow)

There's a tradeoff: chain-of-thought triples your token usage and triples your latency. On a fast task you don't need it. On anything where being wrong matters, you do.

## Critique: use a different model

A common pattern is to ask a model to review its own output. This helps a little, but it has a fundamental limitation that's worth understanding.

Models have systematic blind spots — categories of errors they're prone to, baked in by their training data and architecture. **A model critiquing its own output is looking for failures it's structurally unlikely to produce.** It will catch surface mistakes (typos, formatting) and miss the deeper ones (the assumption it baked in, the edge case its training underrepresented).

The real fix is a *different* model as critic. Different training data, different blind spots. What the first model couldn't see, the second one often can. This pattern scales up to the **advisor pattern** in notebook 09: a fast cheap model generates, a stronger model critiques. You get most of the quality of the stronger model at most of the cost of the cheaper one.

In [6]:
# Pull a second model first:  ollama pull gemma3:4b
GENERATOR = "qwen3.5:9b"
CRITIC    = "gemma3:4b"   # different family — different blind spots

task = "Write a Python function that returns True if a string is a valid email address."

draft = chat(
    [{"role": "user", "content": task}],
    model=GENERATOR,
)

review = chat(
    [{"role": "system",
      "content": "You review code rigorously. Identify bugs, edge cases, security issues. "
                   "Do not assume the code is correct. If you find no issues, say so explicitly."},
     {"role": "user", "content": f"Review this:\n\n{draft}\n\nList specific issues."}],
    model=CRITIC,
    temperature=0.0,
)

print("DRAFT:\n", draft)
print("\nREVIEW (different model):\n", review)

Try this with a real bug — give the generator a known-hard task like 'parse a date string in any format' and watch the critic flag the timezone handling the generator forgot. The critic isn't smarter; it just doesn't share the assumption.

## Structured output

When the next thing in your pipeline needs to parse the response, generic prose is the enemy. Ask for JSON, set temperature to 0, validate the result, retry on failure. Notebook 02 will show how `llm_engines` makes this robust automatically. For now, the manual version:

In [7]:
import json

def extract_invoice(text: str) -> dict | None:
    raw = chat([
        {"role": "system", "content":
         "Extract structured invoice data. Respond with ONE JSON object and nothing else. "
         "Schema: {vendor: string, date: string ISO 8601, amount: number, currency: string}"},
        {"role": "user", "content": text},
    ], temperature=0.0)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Smaller models sometimes wrap JSON in ```json fences. Strip them and retry.
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return None

result = extract_invoice(
    "Invoice from Acme Corp, dated April 1 2026, total: USD 1,250.00 due net 30."
)
print(result)

Notice the retry on JSON-fence wrapping. Smaller models do this constantly. Production code either tolerates it or uses constrained generation (Ollama's `format="json"` parameter, OpenAI's response_format, Anthropic's tool use). `llm_engines` handles all of this for you — but you should see the raw failure mode once.

## What to take into the rest of the course

Five things from this notebook show up in every notebook after this one:

1. **Set temperature explicitly.** Default values vary by provider and silently change behavior.
2. **System prompts are specifications.** Vague specs produce vague output.
3. **Few-shot beats describing the format.** Three examples is the sweet spot.
4. **Chain-of-thought is free accuracy.** Use it on anything where being wrong matters.
5. **A second model is a better critic than the first one.** This becomes the advisor pattern.

## Exercises

1. Take a real piece of work you'd send to a model — code review, data extraction, summarization. Write three system prompts at increasing levels of specificity. Run each. Pick the one you'd actually deploy.

2. Implement a two-model critique loop: generate with model A, critique with model B, incorporate the critique with model A again. On what kinds of tasks does the second pass help? On what kinds does it just add latency?

3. Write `extract_invoice` to handle 10 invoice strings of varying messiness. Measure how often the JSON parse succeeds on the first try. Where it fails, look at the raw output — what is the model doing wrong?

---
**Next:** [01 — Environment Setup](01_environment_setup.ipynb) gets your venv, Ollama, and `ai_tools` packages installed and verified. Then 02 introduces `llm_engines`, which is the abstraction over everything you just did by hand.